In [1]:
!pip install requests jsonschema -q

In [2]:
import os
import json
import re
import requests
import joblib
from jsonschema import validate, ValidationError

In [26]:
import getpass
import os

os.environ["LLM_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

api_key = os.environ["LLM_API_KEY"]

api_key = os.environ["LLM_API_KEY"]

print("API Key Loaded Successfully")

Enter your Groq API Key: ··········
API Key Loaded Successfully


In [9]:
import joblib

model = joblib.load("best_model (1).pkl")

print("Model Loaded Successfully")

Model Loaded Successfully


In [10]:
url = "https://api.groq.com/openai/v1/chat/completions"

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [11]:
system_prompt = "Reply with only the word: hello"

user_prompt = "Say hello."

response = call_llm(system_prompt, user_prompt)

print(response)

Hello.


In [12]:
import re

def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

In [13]:
text1 = "Contact me at test@gmail.com"

text2 = "This is a clean input."

print(has_pii(text1))
print(has_pii(text2))

True
False


In [14]:
schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

In [15]:
system_prompt = """
You are an AI model explanation assistant.

Return ONLY valid JSON.

Output format:

{
"prediction_label":"",
"confidence_level":"",
"top_reason":"",
"second_reason":"",
"next_step":""
}
"""

In [20]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")

# Training ke time jaisa encoding
X = pd.get_dummies(df.drop(columns=["income"]), drop_first=True)

# Model ko jitne columns chahiye utne hi do
expected_cols = model.feature_names_in_

# Missing columns add karo
for col in expected_cols:
    if col not in X.columns:
        X[col] = 0

# Extra columns hatao aur same order me arrange karo
X = X[expected_cols]

sample_inputs = X.iloc[:3]

print(sample_inputs.head())

   fnlwgt  education.num  capital.gain  capital.loss  hours.per.week  \
0   77053              9             0          4356              40   
1  132870              9             0          4356              18   
2  186061             10             0          4356              40   

   workclass_Local-gov  workclass_Never-worked  workclass_Private  \
0                False                   False               True   
1                False                   False               True   
2                False                   False               True   

   workclass_Self-emp-inc  workclass_Self-emp-not-inc  ...  \
0                   False                       False  ...   
1                   False                       False  ...   
2                   False                       False  ...   

   native.country_Puerto-Rico  native.country_Scotland  native.country_South  \
0                       False                    False                 False   
1                       F

In [21]:
for i in range(len(sample_inputs)):

    row = sample_inputs.iloc[[i]]

    pred = model.predict(row)[0]
    proba = model.predict_proba(row)[0].max()

    user_prompt = f"""
Features:
{row.to_dict(orient="records")[0]}

Prediction:
{pred}

Probability:
{proba}
"""

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        continue

    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print("\nRAW RESPONSE:\n")
    print(response)

    try:

        data = json.loads(response.strip())

        validate(
            instance=data,
            schema=schema
        )

        print("Validation: PASS")

    except Exception as e:

        print("Validation: FAIL")
        print(e)


RAW RESPONSE:

{
"prediction_label": "1",
"confidence_level": "0.836344101032974",
"top_reason": "High probability of income > 50K based on education and occupation",
"second_reason": "Presence of a bachelor's degree and a professional specialty occupation",
"next_step": "Further analysis of the individual's work experience and hours worked per week"
}
Validation: PASS

RAW RESPONSE:

{
"prediction_label": "1",
"confidence_level": "0.8691668816644745",
"top_reason": "High probability of income > 50K based on education level and occupation",
"second_reason": "Presence of a bachelor's degree and a high-level occupation",
"next_step": "Further analysis of the individual's work experience and job stability"
}
Validation: PASS

RAW RESPONSE:

{
"prediction_label": "1",
"confidence_level": "0.7713180910982738",
"top_reason": "The individual has a high probability of earning more than $50K due to their education level and occupation.",
"second_reason": "The individual's marital status and na

In [22]:
row = sample_inputs.iloc[[0]]

pred = model.predict(row)[0]
proba = model.predict_proba(row)[0].max()

user_prompt = f"""
Features:
{row.to_dict(orient="records")[0]}

Prediction:
{pred}

Probability:
{proba}
"""

print("Temperature = 0\n")

print(
    call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )
)

print("\n\nTemperature = 0.7\n")

print(
    call_llm(
        system_prompt,
        user_prompt,
        temperature=0.7
    )
)

Temperature = 0

{
"prediction_label": "1",
"confidence_level": "0.836344101032974",
"top_reason": "High probability of income > 50K based on education and occupation",
"second_reason": "Presence of a bachelor's degree and a professional specialty occupation",
"next_step": "Further analysis of the individual's work experience and hours worked per week"
}


Temperature = 0.7

{
"prediction_label": "1",
"confidence_level": "0.836344101032974",
"top_reason": "High probability of earning >50K",
"second_reason": "Strong correlation between education and income",
"next_step": "Collect more data to refine predictions"
}


In [23]:
bad_input = "Email is abc@gmail.com"

good_input = "Income prediction"

print(has_pii(bad_input))

print(has_pii(good_input))

True
False


In [24]:
print("Track C Completed Successfully")

Track C Completed Successfully
